# Step 1: Load and inspect raw ETF files

We load the three Kaggle price files (SPY, TLT, XLF), convert the `Date` column to real dates using a strict format so any malformed date stops the notebook, and sort each table by date. We then check for repeated dates and record each file's first date, last date and number of rows. These checks confirm the files cover the expected period (25 Feb 2005 – 10 Nov 2017) before anything is built on them.

In [ ]:
RAW = 'data/raw/' # relative to the notebook's working folder
import os; print(os.listdir(RAW)) # should list the 4 files

FileNotFoundError: [Errno 2] No such file or directory: 'raw/'

In [ ]:
import os
import pandas as pd

FILES = {"SPY": "spy.us.txt", "TLT": "tlt.us.txt", "XLF": "xlf.us.txt"}

def load_etf(path):
    df = pd.read_csv(path)

    # strict: raises on bad dates, so pandas stops with an error instead of carrying on.
    df["Date"] = pd.to_datetime(df["Date"], format="%Y-%m-%d")
    df = df.sort_values("Date", kind="stable").reset_index(drop=True)

    return df

def summarise(ticker, df):
    dup_dates = df.loc[df["Date"].duplicated(keep=False), "Date"].unique()
    return {"ticker": ticker,
            "start": df["Date"].min().date(),
            "end": df["Date"].max().date(),
            "rows": len(df),
            "n_dup_dates": len(dup_dates)}

data = {t: load_etf(os.path.join(RAW, f)) for t, f in FILES.items()}
summary_df = pd.DataFrame([summarise(t, df) for t, df in data.items()]).set_index("ticker")
summary_df

ModuleNotFoundError: No module named 'pandas'

# Step 2: Clean and sanity-check prices

We drop the `OpenInt` column after confirming it is all zeros, since open interest has no meaning for an ETF. We then flag rows that are physically impossible: a High below the Low, a price of zero or less, or negative volume. These rows are reported, not deleted. Genuine extreme days, such as the 2010 flash crash, are kept, because they are exactly the high-volatility events we want to forecast.

In [ ]:
PRICE_COLS = ["Open", "High", "Low", "Close"]

def clean_and_check(ticker, df):
    # 1. OpenInt: confirm it carries no information, then drop it
    assert (df["OpenInt"] == 0).all(), f"[{ticker}] OpenInt has non-zero values"
    df = df.drop(columns="OpenInt")

    # 2. Sanity checks: one boolean mask per rule (True = impossible row)
    checks = {
        "high_below_low": df["High"] < df["Low"],
        "price_not_positive": (df[PRICE_COLS] <= 0).any(axis=1),
        "negative_volume": df["Volume"] < 0,
    }

    # 3. Report, don't delete: print how many rows break each rule
    for rule, mask in checks.items():
        n_bad = int(mask.sum())
        print(f"[{ticker}] {rule}: {n_bad} rows")
        if n_bad:
            print("   dates:", df.loc[mask, "Date"].dt.strftime("%Y-%m-%d").tolist())

    return df

# Apply to the three tables from step 1
data = {t: clean_and_check(t, df) for t, df in data.items()}

# Step 3: Merge VIX and align to SPY trading days

We add the VIX (series VIXCLS from FRED) and combine all four files into one table with one row per SPY trading day. Where TLT, XLF or the VIX has no value for a SPY day, we carry the last known value forward, never backward, so no row ever uses information from the future. A before/after table shows the percentage of missing values in each column.

In [ ]:
# 3a. Load the VIX from FRED; holidays are "." or blank, so read them as missing
vix = pd.read_csv(RAW + "VIXCLS.csv", na_values=".")
vix = vix.rename(columns={vix.columns[0]: "Date", "VIXCLS": "VIX"})  # date column name varies by download
vix["Date"] = pd.to_datetime(vix["Date"], format="%Y-%m-%d")

# 3b. Prefix TLT/XLF columns so they don't clash with SPY's (e.g. "TLT_Close")
def add_prefix(df, ticker):
    return df.rename(columns={c: f"{ticker}_{c}" for c in df.columns if c != "Date"})

tlt = add_prefix(data["TLT"], "TLT")
xlf = add_prefix(data["XLF"], "XLF")

# 3c. Start from SPY's dates and left-join the others onto them
merged = data["SPY"]
for other in [tlt, xlf, vix]:
    merged = merged.merge(other, on="Date", how="left", validate="one_to_one")
assert len(merged) == len(data["SPY"]), "Merge changed the number of SPY rows"

# 3d. % missing per column BEFORE filling
missing_before = merged.isna().mean() * 100

# 3e. Forward-fill only (never backward): copy the last known value onto gaps
fill_cols = [c for c in merged.columns if c.startswith(("TLT_", "XLF_")) or c == "VIX"]
merged[fill_cols] = merged[fill_cols].ffill()

# 3f. % missing AFTER, side by side with before
missing_after = merged.isna().mean() * 100
missing_table = pd.DataFrame({"before_%": missing_before, "after_%": missing_after}).round(2)
missing_table

# Step 4 — Compute returns, target and memory features (1–3)

# Step 5 — Build the leverage feature (5)

# Step 6 — Build VIX and cross-asset features (4, 6–8)

# Step 7 — Build activity and trend features (9–10)

# Step 8 — Trim incomplete rows, split and save model table

# Step 9 — Set up CV splitter, metrics and prediction export